# DeepParse — full fine-tune of DeepSeek-R1-Distill-Llama-8B on Colab

This notebook reproduces the paper's training procedure end to end:

1. Clone the artifact repo.
2. Install runtime + training extras.
3. Fetch all 16 LogHub-2k corrected datasets.
4. Build the paper-spec training set (50 entropy-greedy examples per system).
5. Fine-tune `deepseek-ai/DeepSeek-R1-Distill-Llama-8B` with LoRA (rank 8, alpha 32, dropout 0.01, AdamW lr=2e-4, batch 8, 25 epochs).
6. Run the GA/PA evaluation through the trained adapter.
7. Download the adapter for offline use.

Recommended runtime: **GPU (A100 / L4 / T4 high RAM)**. Training takes 30-90 min depending on the GPU.

## 0. GPU + environment check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version

## 1. Clone repo + install

In [ ]:
%cd /content
![ -d DeepParse ] || git clone https://github.com/AmirShetaia/DeepParse.git
%cd DeepParse
!pip install -q -e ".[train,test]"
!python -m pytest -q

## 2. Fetch LogHub-2k (corrected) datasets

All 14 systems published in `logpai/loghub-2.0/2k_dataset` (the paper's corrected benchmark) are pulled here. Each system is converted to the directory layout expected by `deepparse.evaluation.eval_runner`.

In [ ]:
!python -m deepparse.tools.fetch_loghub --out artifacts/data
!ls artifacts/data

## 3. Build the paper-spec training set

Algorithm 1 of the paper selects 50 entropy-greedy examples per system. The 16 systems × 50 examples = 800 (some systems yield slightly fewer if alignment fails).

In [ ]:
!python -m deepparse.tools.build_training_set \
    --entropy-k 50 \
    --out artifacts/training/train_paper.jsonl
!wc -l artifacts/training/train_paper.jsonl
!head -1 artifacts/training/train_paper.jsonl

## 4. Fine-tune DeepSeek-R1-Distill-Llama-8B with LoRA

Hyperparameters match the paper Section *LLM Configuration and Fine-Tuning*:

| Hyperparameter | Value |
|---|---|
| Base model | `deepseek-ai/DeepSeek-R1-Distill-Llama-8B` |
| LoRA rank / alpha / dropout | 8 / 32 / 0.01 |
| Optimiser | AdamW |
| Learning rate | 2e-4 (cosine decay) |
| Batch size / grad accum | 8 / 4 |
| Epochs | 25 |
| Precision | bfloat16 |
| Max sequence length | 512 |

In [ ]:
!python -m deepparse.training.finetune \
    --train artifacts/training/train_paper.jsonl \
    --output-dir artifacts/checkpoints/deepparse-r1-8b \
    --epochs 25

## 5. Evaluate GA / PA across all 16 systems via the trained adapter

The evaluator first calls `synth_masks(mode="hf", adapter=…)` to produce a per-system regex bundle, then runs the deterministic Drain parser on the full 2k corpus and scores against the corrected ground truth.

In [ ]:
%%bash
for SYSTEM in $(ls artifacts/data); do
  deepparse synth --dataset $SYSTEM --mode hf \
      --adapter artifacts/checkpoints/deepparse-r1-8b \
      --out artifacts/masks/${SYSTEM}.json
done
deepparse eval --config configs/eval_16_datasets.yaml --deterministic
deepparse table --inputs artifacts/outputs/table_I_ga_pa.csv \
                --out artifacts/outputs/tables/

## 6. Inspect the results table

In [ ]:
import pandas as pd
df = pd.read_csv('artifacts/outputs/table_I_ga_pa.csv')
df

## 7. Download the trained adapter

In [ ]:
!tar -czf deepparse-r1-8b.tar.gz -C artifacts/checkpoints deepparse-r1-8b
from google.colab import files
files.download('deepparse-r1-8b.tar.gz')